# Single spin: Bayesian optimization of a gate

This is similar to the O2B_Single_qubit_gate example, except that it uses Bayesian optimization instead of gradient descent.

In [ ]:
import matplotlib.pyplot as plt

import numpy as np
from jax import Array

from paraqeet.logger import Logger
from paraqeet.optimization_map import OptimizationMap
from paraqeet.quantity import Quantity
from paraqeet.measurement.unitary_fidelity import UnitaryFidelity
from paraqeet.model.drive_operator import DriveOperator
from paraqeet.model.qubit import Qubit
from paraqeet.optimizers.bayesian_optimizer import BayesianOptimizer
from paraqeet.propagation.scipy_expm_goat import ScipyExpmGOAT

from paraqeet.model.closed_system import ClosedSystem

from paraqeet.signal.iq_mixer import IQMixer
from paraqeet.signal.envelopes import ConstantEnvelope

## Setup

We first set up the qubit system we want to control. We set the qubit frequency $\omega_q / 2 \pi$ to be $4.8$ GHz and define the Hamiltonian as $$H(t)=H_\text{drift}+H_c(t)= \frac{\omega_q}{2} \sigma_z + \Omega(t)\sigma_x, $$ where $\Omega(t)$ will be supplied by the generator.

In [ ]:
freq_q = 4.8e9
omega_q = 2 * np.pi * freq_q

controlled_qubit = Qubit(frequency=Quantity(omega_q, 0.8 * omega_q, 1.2 * omega_q, unit="Hz", two_pi=True), drives=[])
model = ClosedSystem(controlled_qubit)

For signal generation, we define a simple cosine shaped tone generator $A \cos(\omega t)$

In [ ]:
t_simu = 3e-9
tone = ConstantEnvelope()
tone.t_final.set_value(t_simu)
gen = IQMixer(envelopes=[tone])

We can inspect the pre-defined parameters with

In [ ]:
params_gen = gen.get_parameters()

In this notebook, we would like to optimize the amplitude `Amplitude` and frequency `lo_freq` if the drive. We add a drive on the qubit.

In [ ]:
freq = 4.8e9 * 2 * np.pi

drive = DriveOperator(gen, is_longitudinal=False)
controlled_qubit = Qubit(frequency=Quantity(freq, 0.8 * freq, 1.2 * freq), drives=[drive])
model = ClosedSystem(controlled_qubit)

Textbook values for implementing an $X$ rotation on this system at a time $T$ would be $\omega=\omega_q$ and $A=\pi/T$. We use some offset from these values as initial guess to demonstrate the optimization procedure.

In [ ]:
params_gen[0].set_value(0.5 * np.pi / t_simu)
params_gen[2].set_value(1.01 * freq)

We select a propagation method, piecewise constant exponentation, and configure an $X$-gate as a target gate. Also we initialize the identity at time $0$.

In [ ]:
prop = ScipyExpmGOAT(model, res=500e9)

times = np.array([0.0, t_simu])

pauli_x = np.array([[0.0, 1.0], [1.0, 0.0]])
pauli_y = np.array([[0.0, -1.0j], [1.0j, 0.0]])
pauli_z = np.array([[1.0, 0.0], [0.0, -1.0]])


prop.set_initial_state(np.identity(2))
gate_fid = UnitaryFidelity(
    propagation=prop,
    gate=pauli_x,
)

In [ ]:
def plot_states():
    """Plot the states."""
    ts = np.linspace(0, t_simu, 1001)
    states = prop.propagate(ts)
    sig = gen.generate_signal(ts)

    _, ax = plt.subplots(2, figsize=(4, 4), sharex=True)
    ax[0].plot(ts / 1e-9, sig)
    ax[0].set_ylabel("Field [MHz]")
    ax[1].plot(ts / 1e-9, np.abs(states[:, 0, 0]) ** 2, label="$P_0$")
    ax[1].plot(ts / 1e-9, np.abs(states[:, 1, 0]) ** 2, label="$P_1$")
    ax[1].set_ylabel("Population")
    ax[-1].set_xlabel("Time [ns]")
    ax[1].grid()
    ax[1].legend()
    plt.show()


plot_states()

As expected, we get a partial transfer and a low fidelity.

In [ ]:
print(f"Gate fidelity: {gate_fid.measure(times)}")

## Custom logger implementation
We define an optimizer and link our fidelity measure as a goal function and the parameters of the cosine tone. We also use a custom logger class to collect all samples that the optimizer takes

In [ ]:
samples = []


class CustomLogger(Logger):
    """Custom logger class definition."""

    def log(self, params: list[Quantity], infid: Array):
        """Log the list of quantities and the fidelity.

        Parameters
        ----------
        params: list[Quantity]
            List of parameters of the system.
        infid: Array
            Inverse of fidelity.

        """
        samples.append((params[0].get_value(), params[1].get_value()))


optmap = OptimizationMap()
optmap.add(gen, [params_gen[0], params_gen[2], params_gen[3]])
opt = BayesianOptimizer(gate_fid, optimizables=optmap, initial_samples=10, iterations=100)
opt.logger = CustomLogger()

In [ ]:
opt.optimize(times)

The plot shows the all the samples that the optimization took in the two-dimensional parameter space. The red dot marks the best value.

In [ ]:
plt.figure()
plt.scatter([s[0] for s in samples[:-1]], [s[1] for s in samples[:-1]], c="blue")
plt.scatter([params_gen[0].get_value()], [params_gen[2].get_value()], c="red", marker="o", s=100)
plt.xlim(params_gen[0].get_min_value()[0], params_gen[0].get_max_value()[0])
plt.ylim(params_gen[2].get_min_value()[0], params_gen[2].get_max_value()[0])
plt.xlabel(params_gen[0].get_name())
plt.ylabel(params_gen[2].get_name())
plt.show()

In [ ]:
plot_states()

We can see from the plot and optimizer output that we have found good controls.

In [ ]:
print(f"Gate fidelity: {gate_fid.measure(times)}")